In [ ]:
import zipfile

# Path to the zip file you want to extract
zip_file_path = "Group_16.zip"

# Path to the directory where you want to extract the contents
destination_dir = ""

# Extract the archive while preserving the directory structure
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(destination_dir)

print(f"Successfully unzipped to '{destination_dir}' with directories intact!")

Successfully unzipped to '' with directories intact!


In [ ]:
import os
import copy
import json
import numpy as np
import random
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import SGD, Adam, Adagrad, RMSprop
from torchvision import datasets, transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")


LEARNING_RATE = 0.001
MOMENTUM = 0.9
epochs = 10000
THRESHOLD = 1e-4

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


ACTIVATION = "tanh"

criterion = nn.CrossEntropyLoss()

ARCHITECTURES = {
    5: {
        "config_1": [128, 64, 32, 16, 8],
        "config_2": [256, 128, 64, 32, 16],
        "config_3": [512, 256, 128, 64, 32]
    }
}

Using GPU: Tesla T4


In [ ]:
data_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(
    'Group_16/train',
    transform=data_transforms
)

val_data = datasets.ImageFolder(
    'Group_16/val',
    transform=data_transforms
)

test_data = datasets.ImageFolder(
    'Group_16/test',
    transform=data_transforms
)

print(train_data.class_to_idx)
print(val_data.class_to_idx)
print(test_data.class_to_idx)

assert train_data.class_to_idx == val_data.class_to_idx
assert train_data.class_to_idx == test_data.class_to_idx

print("Class mapping verified:", train_data.class_to_idx)

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Test samples:", len(test_data))

image, label = train_data[0]

print("Image shape:", image.shape)

{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
{'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
Class mapping verified: {'1': 0, '5': 1, '6': 2, '7': 3, '9': 4}
Training samples: 11385
Validation samples: 3795
Test samples: 3795
Image shape: torch.Size([1, 28, 28])


#Optimizers and corresponding configs that will be used for each model

In [ ]:
def sgd(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE)

def sgd_general_delta(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE,
             momentum=MOMENTUM)

def sgd_nag(model):
  return SGD(model.parameters(),
             lr=LEARNING_RATE,
             momentum=MOMENTUM,
             nesterov=True)

def adagrad(model):
  return Adagrad(model.parameters(),
                 lr=LEARNING_RATE,
                 eps=1e-08
                 )

def rmsprop(model):
  return RMSprop(model.parameters(),
                lr=LEARNING_RATE,
                eps=1e-08,
                alpha=0.99,
                )

def adam(model):
  return Adam(model.parameters(),
              lr=LEARNING_RATE,
              betas=(0.9, 0.999),
              eps=1e-08
              )

optimizers={
    'sgd': sgd,
    'vanillaGD':sgd,
    'sgd_general_delta': sgd_general_delta,
    'sgd_nag': sgd_nag,
    'adagrad': adagrad,
    'rmsprop': rmsprop,
    'adam':adam
}

optimizer_config={
    'sgd':{
        'batch_size':1
    },
    'vanillaGD':{
        'batch_size':len(train_data)
    },
    'sgd_general_delta':{
        'batch_size':1
    },
    'sgd_nag':{
        'batch_size':1
    },
    'adagrad':{
        'batch_size':len(train_data)
    },
    'rmsprop':{
        'batch_size':len(train_data)
    },
    'adam':{
        'batch_size':1
    }
}

In [ ]:
def build_model(hidden_nodes):

    layers = [nn.Flatten()]

    for i in range(len(hidden_nodes)):

        if i == 0:
            layers.append(
                nn.Linear(784, hidden_nodes[i])
            )
        else:
            layers.append(
                nn.Linear(
                    hidden_nodes[i - 1],
                    hidden_nodes[i]
                )
            )

        layers.append(nn.Tanh())

    layers.append(
        nn.Linear(hidden_nodes[-1], 5)
    )

    return nn.Sequential(*layers)

In [ ]:
os.makedirs("initial", exist_ok=True)

for architecture in ARCHITECTURES:

    for config_name, hidden_nodes in ARCHITECTURES[architecture].items():

        output_path = (
            f"initial/model_{architecture}_{config_name}.pth"
        )

        if os.path.exists(output_path):

            print(f"Already exists: {output_path}")

        else:

            model = build_model(hidden_nodes)

            torch.save(
                model.state_dict(),
                output_path
            )

            print(f"Created: {output_path}")

Created: inital/model_5_tanh.pth
Created: inital/model_5_sigmoid.pth


In [ ]:
best_local_val_accuracy = -1
best_local_model = None
best_local_optimizer = None
best_local_config = None

In [ ]:
def calculate_average_error(model, data):

    data_loader = DataLoader(
        data,
        batch_size=128,
        shuffle=False,
        pin_memory=(device.type == "cuda")
    )

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for x, y in data_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            output = model(x)

            loss = criterion(output, y)

            batch_size = y.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples

In [ ]:
def classification_accuracy(model, data):

    data_loader = DataLoader(
        data,
        batch_size=128,
        shuffle=False,
        pin_memory=(device.type == "cuda")
    )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in data_loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            output = model(x)

            prediction = torch.argmax(
                output,
                dim=1
            )

            correct += (
                prediction == y
            ).sum().item()

            total += y.size(0)

    return 100 * correct / total

In [ ]:
model_histories = {}


for architecture in ARCHITECTURES:

    model_histories[architecture] = {}

    for config_name, hidden_nodes in ARCHITECTURES[architecture].items():

        model_histories[architecture][config_name] = {}

        print("=" * 80)
        print(
            f"Architecture: {architecture} hidden layers | "
            f"Configuration: {config_name} | "
            f"Neurons: {hidden_nodes}"
        )
        print("=" * 80)


        for optimizerName, config in optimizer_config.items():

            print("-" * 80)
            print(
                f"Architecture: {architecture} | "
                f"Configuration: {config_name} | "
                f"Neurons: {hidden_nodes} | "
                f"Optimizer: {optimizerName}"
            )
            print("-" * 80)


            # Build model
            model = build_model(hidden_nodes)


            # Load SAME initial weights for every optimizer
            model.load_state_dict(
                torch.load(
                    f"initial/model_{architecture}_{config_name}.pth",
                    map_location=device
                )
            )

            model = model.to(device)


            # Initial error BEFORE any optimizer update
            initial_error = calculate_average_error(
                model,
                train_data
            )


            # Create optimizer
            optimizer = optimizers[optimizerName](model)


            # Same shuffle seed for each optimizer
            generator = torch.Generator()
            generator.manual_seed(SEED)


            train_loader = DataLoader(
                train_data,
                batch_size=config["batch_size"],
                shuffle=True,
                generator=generator,
                pin_memory=(device.type == "cuda")
            )


            errors = [initial_error]


            # Training
            for epoch in range(epochs):

                model.train()

                training_loss = 0.0
                total_samples = 0


                for x, y in train_loader:

                    x = x.to(
                        device,
                        non_blocking=True
                    )

                    y = y.to(
                        device,
                        non_blocking=True
                    )


                    optimizer.zero_grad()

                    output = model(x)

                    loss = criterion(output, y)

                    loss.backward()

                    optimizer.step()


                    batch_size = y.size(0)

                    training_loss += (
                        loss.item() * batch_size
                    )

                    total_samples += batch_size


                # Average training error
                average_error = (
                    training_loss /
                    total_samples
                )

                errors.append(average_error)


                if (
                    (epoch + 1) % 10 == 0
                    or epoch == 0
                ):

                    print(
                        f"Epoch {epoch + 1}: "
                        f"Error = {average_error:.8f}"
                    )


                # Convergence criterion
                error_difference = abs(
                    errors[-1] - errors[-2]
                )


                if epoch>=2 and error_difference < THRESHOLD:

                    print(
                        f"Converged at epoch {epoch + 1}"
                    )

                    break


            # Final training accuracy
            final_train_accuracy = (
                classification_accuracy(
                    model,
                    train_data
                )
            )


            # Final validation accuracy
            final_val_accuracy = (
                classification_accuracy(
                    model,
                    val_data
                )
            )


            # Save history
            model_histories[
                architecture
            ][
                config_name
            ][
                optimizerName
            ] = {

                "hidden_nodes":
                    hidden_nodes,

                "initial_error":
                    initial_error,

                "errors":
                    errors,

                "epochs":
                    len(errors) - 1,

                "final_error":
                    errors[-1],

                "final_train_accuracy":
                    final_train_accuracy,

                "final_val_accuracy":
                    final_val_accuracy
            }


            # Find best model on this laptop
            if final_val_accuracy > best_local_val_accuracy:

                best_local_val_accuracy = final_val_accuracy

                best_local_config = config_name

                best_local_optimizer = optimizerName

                best_local_model = copy.deepcopy(
                    model.state_dict()
                )


            print(
                f"Final Error: "
                f"{errors[-1]:.8f}"
            )

            print(
                f"Final Train Accuracy: "
                f"{final_train_accuracy:.2f}%"
            )

            print(
                f"Final Validation Accuracy: "
                f"{final_val_accuracy:.2f}%"
            )

            print(
                f"Converged Epochs: "
                f"{len(errors) - 1}"
            )

            print()

Activation: tanh | Architecture: 5 | Optimizer: sgd
Epoch 1: Error = 1.22026370
Epoch 10: Error = 0.09233413
Epoch 20: Error = 0.04333185
Epoch 30: Error = 0.01950521
Epoch 40: Error = 0.01023335


In [ ]:
architecture_id = list(ARCHITECTURES.keys())[0]

torch.save(
    {
        "architecture": architecture_id,

        "config": best_local_config,

        "hidden_nodes":
            ARCHITECTURES[
                architecture_id
            ][
                best_local_config
            ],

        "activation": "tanh",

        "optimizer":
            best_local_optimizer,

        "validation_accuracy":
            best_local_val_accuracy,

        "model_state_dict":
            best_local_model
    },

    f"best_model_architecture_{architecture_id}.pth"
)

In [ ]:
architecture_id = list(ARCHITECTURES.keys())[0]

results = {

    "architecture":
        architecture_id,

    "activation":
        "tanh",

    "neuron_configurations":
        ARCHITECTURES[architecture_id],

    "learning_rate":
        LEARNING_RATE,

    "momentum":
        MOMENTUM,

    "threshold":
        THRESHOLD,

    "epochs_limit":
        epochs,

    "optimizers":
        list(optimizer_config.keys()),

    "histories":
        model_histories
}

In [ ]:
json_filename = (
    f"model_history_architecture_{architecture_id}.json"
)

with open(json_filename, "w") as f:

    json.dump(
        results,
        f,
        indent=4
    )

print(
    f"Saved results to {json_filename}"
)

In [ ]:
import matplotlib.pyplot as plt

architecture_id = list(ARCHITECTURES.keys())[0]

for config_name, hidden_nodes in ARCHITECTURES[architecture_id].items():

    plt.figure(figsize=(10, 6))

    for optimizerName in optimizer_config:

        errors = model_histories[
            architecture_id
        ][
            config_name
        ][
            optimizerName
        ][
            "errors"
        ]

        plt.plot(
            range(len(errors)),
            errors,
            label=optimizerName
        )

    plt.xlabel("Epochs")
    plt.ylabel("Average Training Error")

    plt.title(
        f"Architecture {architecture_id} | "
        f"{config_name} | "
        f"Neurons: {hidden_nodes} | "
        f"Tanh"
    )

    plt.legend()

    plt.grid(True)

    plt.tight_layout()

    plt.show()